# Boston crossing-density map

Self-contained pipeline: builds the ZIP study area, pulls pedestrian crossings and OSM grass, clips both to the study area, and writes a standalone interactive HTML map with the data baked in (kernel-density surface + hexbins, boundary/water clip, grass overlay).

Run top to bottom. The only thing you may want to edit is the **Config** cell.

In [1]:
!pip install geopandas osmnx pygris shapely pandas pyogrio -q

## Config

In [2]:
TARGET_ZIPS = [
    "02108","02109","02110","02111","02113","02114","02115","02116",
    "02118","02119","02120","02121","02122","02124","02125","02127",
    "02128","02129","02130","02133","02134","02135","02138","02139",
    "02141","02142","02163","02199","02203","02205","02210","02215",
    "02445","02446","02458","02459","02467","02472",
]

# Where crossings come from:
#   "osm"  -> OpenStreetMap highway=crossing nodes (marked + unmarked, no download needed)
#   "file" -> a GeoJSON you downloaded (e.g. MassGIS Crosswalks; marked-only, from imagery)
CROSSWALK_SOURCE = "osm"
CROSSWALK_FILE   = "crosswalks.geojson"   # used only when CROSSWALK_SOURCE == "file"

# Grass/green tags. landuse=grass is the literal grass surface but includes medians/verges
# and misses park lawns. Broaden for "walkable green space" if that is what you want, e.g.:
#   {"landuse": ["grass","recreation_ground"], "leisure": ["park","pitch","garden"]}
GRASS_TAGS = {
    "landuse":  ["grass", "meadow", "village_green", "recreation_ground", "cemetery"],
    "natural":  ["grassland"],
    "leisure":  ["park", "garden", "pitch", "common", "playground", "dog_park"],
}

OUT_HTML = "boston_crosswalk_density.html"

## Imports

In [3]:
import json
import geopandas as gpd
import osmnx as ox
from shapely.ops import unary_union
from shapely.geometry import mapping

## Study area (ZIP boundary)
Pulls the 2020 cartographic-boundary ZCTAs via `pygris` (shoreline-clipped, so this doubles as the water mask). Falls back to a local `cb_2020_us_zcta520_500k.shp` if `pygris` is unavailable — with `SHAPE_RESTORE_SHX` on, but note that only recovers the index, not a missing `.dbf`.

In [4]:
try:
    from pygris import zctas
    zcta_all = zctas(year=2020, cb=True, cache=True)
except Exception as e:
    print("pygris unavailable, reading local shapefile instead:", e)
    import pyogrio
    pyogrio.set_gdal_config_options({"SHAPE_RESTORE_SHX": "YES"})
    zcta_all = gpd.read_file("cb_2020_us_zcta520_500k.shp")

zip_col = next((c for c in ["ZCTA5CE20","ZCTA5CE10","GEOID20","GEOID10"] if c in zcta_all.columns), None)
assert zip_col, f"No ZIP column found. Columns present: {list(zcta_all.columns)}"

target = zcta_all[zcta_all[zip_col].isin(TARGET_ZIPS)].to_crs(4326)
assert len(target) > 0, "No ZCTAs matched TARGET_ZIPS \u2014 check the ZIP column / values"
print(f"matched {len(target)} of {len(TARGET_ZIPS)} target ZIPs on column {zip_col!r}")

study_area = unary_union(target.geometry)
BOUNDARY = {"type": "Feature", "properties": {}, "geometry": mapping(study_area)}

matched 38 of 38 target ZIPs on column 'ZCTA5CE20'


## Pedestrian crossings
Collapses each crossing to one point `[lat, lng]`. OSM crossings come back as nodes already; a file source is clipped to the study area and reduced to representative points.

In [5]:
if CROSSWALK_SOURCE == "osm":
    cw = ox.features_from_polygon(study_area, tags={"highway": "crossing"})
    cw = cw[cw.geometry.geom_type == "Point"]
    POINTS = [[float(g.y), float(g.x)] for g in cw.geometry]
else:
    cw = gpd.read_file(CROSSWALK_FILE).to_crs(4326)
    cw = gpd.clip(cw, study_area)
    reps = cw.geometry.representative_point()
    POINTS = [[float(g.y), float(g.x)] for g in reps]

print(len(POINTS), "crossing points")
assert POINTS, "No crossings found \u2014 check the source / study area"

22451 crossing points


## Walkable grass / green space
OSM polygons matching `GRASS_TAGS`, cleaned, and clipped to the study area so parks are cut at the boundary (and the shoreline). `buffer(0)` repairs invalid rings that would otherwise be dropped.

In [9]:
try:
    grass = ox.features_from_polygon(study_area, tags=GRASS_TAGS)
    grass = grass[grass.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
    grass["geometry"] = grass.buffer(0)                    # repair invalid geometries
    grass = grass[~grass.geometry.is_empty]
    grass = gpd.clip(grass, study_area)
    grass = grass[grass.geom_type.isin(["Polygon", "MultiPolygon"])]
    GRASS = json.loads(grass[["geometry"]].reset_index(drop=True).to_json())
    print(len(grass), "grass polygons")
except Exception as e:
    print("grass fetch failed, continuing without it:", e)
    GRASS = {"type": "FeatureCollection", "features": []}

MIN_AREA_M2 = 150   # kills median/verge slivers

g = grass  # your cleaned, clipped GeoDataFrame

# 1) publicly walkable only — drop private/gated
if "access" in g.columns:
    g = g[~g["access"].isin(["private", "no", "permit"])]

# 2) drop clearly non-grass surfaces (mostly affects pitches/plazas)
hard = ["asphalt","concrete","paved","paving_stones","artificial_turf",
        "tartan","clay","sand","rubber","acrylic","dirt","gravel"]
if "surface" in g.columns:
    g = g[~g["surface"].isin(hard)]

# 3) drop tiny slivers by real area (project first)
g = g[g.to_crs(26986).area.values >= MIN_AREA_M2]

5292 grass polygons


## Build the HTML map

In [10]:
TEMPLATE_B64 = "..."

In [11]:
import base64
from IPython.display import FileLink

template = base64.b64decode(TEMPLATE_B64).decode("utf-8")
html = (template
        .replace("%%POINTS%%",   json.dumps(POINTS))
        .replace("%%BOUNDARY%%", json.dumps(BOUNDARY))
        .replace("%%GRASS%%",    json.dumps(GRASS)))

with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print(f"wrote {OUT_HTML}  ({len(html)/1024:.0f} KB, {len(POINTS)} crossings)")
FileLink(OUT_HTML)

wrote boston_crosswalk_density.html  (3870 KB, 22451 crossings)


c:\Users\Sam\WalkSensePlace\boston_crosswalk_density.html